In [1]:
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import os
import re
import json
#from tqdm import tqdm
#import time
import os
from openai import OpenAI
import tiktoken

api_key = os.environ["OPENAI_API_KEY"]
org_key = os.environ.get("OPENAI_ORG_ID")

In [ ]:
df_elig_new = pd.read_csv('paper_collection/WoS_251031_eligible_new.csv')


In [ ]:
def format_apa(row):
    # Get and clean authors from the cell (split on semicolon, remove extra spaces)
    authors = row['Author Full Names']
    authors_list = [author.strip() for author in authors.split(';') if author.strip()]
    
    # Format authors according to APA rules:
    if len(authors_list) == 0:
        authors_formatted = ""
    elif len(authors_list) == 1:
        authors_formatted = authors_list[0]
    else:
        # Join all but the last author with commas, and add " & " before the last
        authors_formatted = ", ".join(authors_list[:-1]) + " & " + authors_list[-1]


        
    
    # Grab the other fields
    publication_year = row['Publication Year']
    article_title = row['Article Title']
    source_title = row['Source Title']  # APA usually uses the journal title as is.
    
    # Construct the APA style reference
    reference = f"{authors_formatted} ({publication_year}). {article_title}. {source_title}."
    return reference

# Apply the function row-wise to generate the new column 'reference_APA'
df_elig_new['reference_APA'] = df_elig_new.apply(format_apa, axis=1)


In [ ]:
df_pgg

In [10]:
df_pgg = pd.read_csv('input/pgg_validation_basePrompt.csv')

def append_prompt(dataframe, augmented_text=''):
    messages = [ make_predict_prompt( make_config(row), augmented_text=augmented_text ) for i, row in dataframe.iterrows() ]
    #dataframe['prompt_'+column] = messages
    return messages

In [11]:
system_prompt = f"""We have conducted multiple public goods game experiments with varying experimental designs, to measure the effect of punishment in cooperative settings under various environments.
Your task is to predict how enabling a punishment mechanism to a specific game changes the ***efficiency*** compared to the same game with punishment disabled.
According to our experiments, whether punishment increases efficiency or not is highly dependent on a lot of dimensions in experiment design, and it is your job to navigate this heterogeneity and make accurate predictions.

***Efficiency*** is the ratio between the game players' behavior and that of a fully-cooperative group (i.e. a group in which all members contribute their full endowment in every round)
In other words, efficiency measures how close a group's total payoff is, compared to that of a group that always cooperates (i.e. always contributes the entire endowment, and benefits maximally from the multiplier). 
An efficiency value of 100% means that a group earned the same amount of coins as a hypothetical group that always cooperated.
            
For example, let's say a game has 5 players playing 10 rounds where 20 coins are given to each player per round and the multiplier for each contributed coin is 3.
In this case, the earning of a hypothetical "always cooperating" group is 5*10*20*3=3000 coins, while the earning of a hypothetical "never cooperating" group is 1000 coins.
Hence, the efficiency is 100% for the always cooperating group and 33% for the never cooperating group.

Your output should strictly be a prediction value with integer only (e.g., 33% should output 33 and nothing else)."""

def make_predict_prompt(config, augmented_text=''):
        closing =   f"""Now, predict the efficiency of the game below when punishment is to be enabled.
### Game Information ###
{config}

You predict that enabling punishment will cause the efficiency percentage to change to (output should be an integer and nothing else):
"""
        return augmented_text + closing
        
def make_config(cd): # cd: configuration dictionary
    game_structure = f"""
***The efficiency of this game with punishment disabled was:*** {int(round(100 * cd['efficiency_np'], 0))}%

[CONFIGURATION]

*** Game Structure ***
Number of players: {int(cd['CONFIG_playerCount'])}
Number of rounds: {int(cd['CONFIG_numRounds'])}
Is chat enabled among players?: {bool(cd['CONFIG_chat'])}
Is the contribution "all or nothing" i.e., binary instead of continuous?: {bool(cd['CONFIG_allOrNothing'])}
Is contribution the default i.e., does each player's endowment start in the public fund for them to opt-out?: {bool(cd['CONFIG_defaultContribProp'])}

*** Monetary Stakes ***
Marginal per capita return (MPCR): {cd['CONFIG_MPCR']}

*** Peer Incentives *** 
    """

    punishment = f"""
Punishment cost to impose a single unit of punishment: {int(cd['CONFIG_punishmentCost'])} coin(s)
Punishment impact (number of coins deducted from the punished player per coin spent punishing): {float(cd['CONFIG_punishmentTech'])}
    """
        # should it be punishment Tech? No magnitude is fine
    reward = f"""
Reward cost to grant a single unit of reward: {int(cd['CONFIG_rewardCost'])} coin(s)
Reward impact (the coins awarded to a player per coin spent rewarding): {float(cd['CONFIG_rewardTech'])}
    """

    information_display = f"""
*** Information Display ***
Is the number of rounds known to players (do they know when the game ends)?: {bool(cd['CONFIG_showNRounds'])}
Are peer outcomes shown (do players know how much their peers gained at the end of each round)?: {bool(cd['CONFIG_showOtherSummaries'])}"""

    information_punishment = f"""
When a player is punished/rewarded, are the punishers/rewarders known?: {bool(cd['CONFIG_showPunishmentId'])}
    """


    no_reward = f"""
Reward mechanism is not enabled.
    """

    
    if cd['CONFIG_rewardExists'] == True:
        return game_structure + punishment + reward + information_display + information_punishment
    else:
        return game_structure + punishment + no_reward + information_display + information_punishment

In [12]:
prompts_baseline = append_prompt(df_pgg)

In [13]:
prompt_summary = """Below is a research synthesis report across academic papers, discussing how the 15 parameters are predicted to affect the punishment treatment effect on efficiency.
Make predictions based faithfully on implications from the synthesis."""
prompt_summary_multiple = """Below are research synthesis reports across academic papers, discussing how the 15 parameters are predicted to affect the punishment treatment effect on efficiency.
Make predictions based faithfully on implications from the syntheses."""
prompt_summary_individual = """Below is an analysis report of an academic paper, discussing how the 15 parameters are predicted to affect the punishment treatment effect on efficiency.
Make predictions based faithfully on implications from the paper."""
prompt_summary_learn = """Below is a list of experiments conducted in the past, outlining how the 15 parameters change the efficiency and the punishment treatment effect on it.
Make predictions based on what you can learn from this."""
prompt_abstract = """Below are abstracts from academic papers discussing the effect of punishment on cooperative games.
Make predictions based faithfully on implications from the paper."""
prompt_abstract_individual = """Below is an abstract of an academic paper discussing the effect of punishment on cooperative games.
Make predictions based faithfully on implications from the paper."""
prompt_fulltext = """Below are full texts of academic papers discussing the effect of punishment on cooperative games.
Make predictions based faithfully on implications from the paper."""
prompt_full_individual = """Below is the full text of an academic paper discussing the effect of punishment on cooperative games.
Make predictions based faithfully on implications from the paper."""
prompt_abstract_hybrid = """Below are abstracts from academic papers, followed by a synthesis report from these papers discussing how the 15 parameters are predicted to affect the punishment treatment effect on efficiency.
Make predictions based faithfully on implications from the paper."""
prompt_abstract_hybrid_individual = """Below is the abstract of an academic paper, followed by an analysis report discussing how the 15 parameters are predicted to affect the punishment treatment effect on efficiency.
Make predictions based faithfully on implications from the paper."""
prompt_fulltext_hybrid = """Below are full texts of academic papers, followed by a synthesis report from these papers discussing how the 15 parameters are predicted to affect the punishment treatment effect on efficiency.
Make predictions based faithfully on implications from the paper."""
prompt_fulltext_hybrid_individual = """Below is the full text of an academic paper, followed by an analysis report discussing how the 15 parameters are predicted to affect the punishment treatment effect on efficiency.
Make predictions based faithfully on implications from the paper."""
def append_from_text(text, messages, instruction=prompt_summary): # load the full paper and append to the prompt
    appended_messages = list()
    try:
        content = text
        
        for m in messages: # append
            if instruction == prompt_summary:
                appended = f"""{instruction}
        ----------Synthesis Report Starts----------
        
        {content}

        ----------Synthesis Report Ends----------
        {m}"""
                appended_messages.append(appended)
            elif instruction == prompt_summary_individual:
                appended = f"""{instruction}
        ----------Report Starts----------
        
        {content}

        ----------Report Ends----------
        {m}"""
                appended_messages.append(appended)
            elif instruction == prompt_summary_learn:
                appended = f"""{instruction}
        ----------Experiments Starts----------
        
        {content}

        ----------Experiments Ends----------
        {m}"""
                appended_messages.append(appended)
            elif instruction in [prompt_abstract_individual, prompt_full_individual, prompt_abstract_hybrid_individual, prompt_fulltext_hybrid_individual]:
                appended = f"""{instruction}
        ----------Paper Starts----------
        
        {content}

        ----------Paper Ends----------
        {m}"""
                appended_messages.append(appended)

            else:
                appended = f"""{instruction}
        {content}
        {m}"""
                appended_messages.append(appended)
        return appended_messages
        
    
    except Exception as e:
        print(f"Error processing augmentation text: {str(e)}")
 

In [14]:
def create_prediction_batch_json(jsonl_summaries=None, text_summaries=None, custom_ids=None, model='gpt-4.1-2025-04-14', prompts = prompts_baseline, instruction = prompt_summary_individual, input_json = True):
    # different from the one above. be careful.
    if input_json:
        custom_ids_elig = set(df_elig['custom_id'])
        requests = list()
        for j in jsonl_summaries:
            if j['custom_id'] in custom_ids_elig:
                augmented_prompts = append_from_text(text=j['response']['body']['choices'][0]['message']['content'], messages=prompts, instruction=instruction)
                for n, p in enumerate(augmented_prompts):
                    requests.append(
                        {"custom_id": f"{j['custom_id'][:-3]}/Q{n+1}",
                        "method": "POST",
                        "url": "/v1/chat/completions",
                        "body": {"model": model,  "messages": [ {"role":"system", "content": system_prompt},
                        {"role": "user",
                    "content": p}],
                    "logprobs": True,
                    "top_logprobs": 20,
                    "temperature": 0}}
                    )
    else:
        requests = list()
        for i, t in enumerate(text_summaries):
            augmented_prompts = append_from_text(text=t, messages=prompts, instruction=instruction)
            for n, p in enumerate(augmented_prompts):
                requests.append(
                    {"custom_id": f"{custom_ids[i][:-3]}/Q{n+1}", # IED summary for this purpose
                    "method": "POST",
                    "url": "/v1/chat/completions",
                    "body": {"model": model,  "messages": [ {"role":"system", "content": system_prompt},
                    {"role": "user",
                "content": p}],
                "logprobs": True,
                "top_logprobs": 20,
                "temperature": 0}}
                )
    return requests

In [ ]:
# abstract
with open(f"OpenAI_batch_input/prediction_251105_individual_abstract_41.jsonl", "w", encoding="utf-8") as jsonl_file:
    for entry in create_prediction_batch_json(text_summaries=list(df_elig_new['Abstract']), custom_ids=list(df_elig_new['custom_id']), instruction=prompt_abstract_individual, model='gpt-4.1-2025-04-14', input_json=False):
        jsonl_file.write(json.dumps(entry, ensure_ascii=False) + "\n")

### Handle Error

# Full Paper

In [ ]:
custom_ids_elig = set(df_elig_new['custom_id'])
full_texts_elig = list()
for custom_id in df_elig_new['custom_id']:
    with open(f'papers_markdown/{custom_id}', "r", encoding="utf-8") as md_file:
        full_texts_elig.append( md_file.read() )

In [ ]:
import math
# -------------------------------------------------------------
# 1)  Materialise the generator once so we know how many rows
# -------------------------------------------------------------
entries = list(
    create_prediction_batch_json(
        text_summaries = full_texts_elig,
        custom_ids     = list(df_elig_new["custom_id"]),
        instruction    = prompt_full_individual,
        model          = "gpt-4.1-2025-04-14",
        input_json     = False,
    )
)

# -------------------------------------------------------------
# 2)  Split into three contiguous segments
# -------------------------------------------------------------
chunk_size = math.ceil(len(entries) / 6)     # ensures ≤1-row size difference

base = "OpenAI_batch_input/prediction_251105_individual_fulltext_41"
os.makedirs(os.path.dirname(base), exist_ok=True)

for i in range(6):                           # 0, 1, 2
    start = i * chunk_size
    end   = start + chunk_size
    with open(f"{base}-{i+1}.jsonl", "w", encoding="utf-8") as f:
        for entry in entries[start:end]:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

### Handle Error

# Reports

In [28]:
df_elig = pd.read_csv('paper_collection/WoS_251031_eligible.csv')
df_elig_new = df_elig[df_elig['collected_251031']]

In [30]:
df_elig_new['o1_summary'].isna().sum()

0

In [31]:
# report
with open(f"OpenAI_batch_input/prediction_251105_individual_report.jsonl", "w", encoding="utf-8") as jsonl_file:
    for entry in create_prediction_batch_json(text_summaries=list(df_elig_new['o1_summary']), custom_ids=list(df_elig_new['custom_id']),
                                              instruction=prompt_summary_individual, model='gpt-4.1-2025-04-14', input_json=False):
        jsonl_file.write(json.dumps(entry, ensure_ascii=False) + "\n")

# Multiple Abstracts

In [2]:
df_elig = pd.read_csv('paper_collection/WoS_251031_eligible.csv')

In [ ]:
df_elig

In [ ]:
df_elig

In [7]:
text_abstract = "---Abstracts Start---"
for i, row in df_elig.iterrows():
    text_abstract += f"""
### Paper {i} ###
{row['reference_APA']}
{row['Abstract']}
"""
text_abstract += "---Abstracts End---"

In [16]:
with open(f"OpenAI_batch_input/prediction_251105_abstracts_41.jsonl", "w", encoding="utf-8") as jsonl_file:
    for entry in create_prediction_batch_json(
            text_summaries = [ text_abstract ],
            custom_ids     = [ 'ALL.md' ],
            instruction    = prompt_abstract,
            model          = "gpt-4.1-2025-04-14",
            input_json     = False,
        ):
        jsonl_file.write(json.dumps(entry, ensure_ascii=False) + "\n")

Filtered collection abstract

In [32]:
PATH_MAP    = "paper_collection/collection_mapping_251110.json"          # {collection: [paper-row idx]}
with open(PATH_MAP, encoding="utf-8") as fh:
    raw_map = json.load(fh)

#n_rows = len(df_d)
coll_map = {lab: [int(x) for x in raw if str(x).isdigit() and 0 <= int(x)]
            for lab, raw in raw_map.items()}

In [34]:
df_elig = pd.read_csv('paper_collection/WoS_251031_eligible_design.csv')

In [38]:
filtered_abstracts = list()
for k, v in coll_map.items():
    text_abstract = "---Abstracts Start---"
    for i, row in df_elig.iloc[v].iterrows():
        text_abstract += f"""
    ### Paper {i} ###
    {row['reference_APA']}
    {row['Abstract']}
    """
    text_abstract += "---Abstracts End---"
    filtered_abstracts.append(text_abstract)

In [39]:
len(filtered_abstracts)

619

In [55]:
with open(f"OpenAI_batch_input/prediction_251110_abstracts_41.jsonl", "w", encoding="utf-8") as jsonl_file:
    for entry in create_prediction_batch_json(
            text_summaries = filtered_abstracts,
            custom_ids     = [ k+".md" for k in coll_map.keys()],
            instruction    = prompt_abstract,
            model          = "gpt-4.1-2025-04-14",
            input_json     = False,
        ):
        jsonl_file.write(json.dumps(entry, ensure_ascii=False) + "\n")

In [51]:
len(list(coll_map.keys()))

619

In [56]:
import math

entries = list(
    create_prediction_batch_json(
            text_summaries = filtered_abstracts,
            custom_ids     = [ k+".md" for k in coll_map.keys()],
            instruction    = prompt_abstract,
            model          = "gpt-4.1-2025-04-14",
            input_json     = False,
        )
)

# -------------------------------------------------------------
# 2)  Split into three contiguous segments
# -------------------------------------------------------------
chunk_size = math.ceil(len(entries) / 3)     # ensures ≤1-row size difference

base = "OpenAI_batch_input/prediction_251110_abstracts_41"
os.makedirs(os.path.dirname(base), exist_ok=True)

for i in range(3):                           # 0, 1, 2
    start = i * chunk_size
    end   = start + chunk_size
    with open(f"{base}-{i+1}.jsonl", "w", encoding="utf-8") as f:
        for entry in entries[start:end]:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

In [17]:
with open(f"OpenAI_batch_input/prediction_251105_abstracts_41.jsonl", "r", encoding="utf-8") as jsonl_file:
    jsonl_all_abstract = [json.loads(line) for line in jsonl_file]  # Load each line as a dictionary

In [24]:
print(j['body']['messages'][-1]["content"][-10000:])

the attempts to deceive that arise in cooperative endeavors may be too costly to make cooperation worthwhile.

### Paper 1391 ###
Baier, Alexandra, Balafoutas, Loukas & Jaber-Lopez, Tarek (2023). Ostracism and theft in heterogeneous groups. EXPERIMENTAL ECONOMICS.
Ostracism, or exclusion by peers, has been practiced since ancient times as a severe form of punishment against transgressors of laws or social norms. The purpose of this paper is to offer a comprehensive analysis on how ostracism affects behavior and the functioning of a social group. We present data from a laboratory experiment, in which participants face a social dilemma on how to allocate limited resources between a productive activity and theft, and are given the opportunity to exclude members of their group by means of majority voting. Our main treatment features an environment with heterogeneity in productivity within groups, thus creating inequalities in economic opportunities and income. We find that exclusion is an 

In [ ]:
# abstract
with open(f"OpenAI_batch_input/prediction_251105_abstracts_41.jsonl", "w", encoding="utf-8") as jsonl_file:
    for entry in create_prediction_batch_json(text_summaries=list(df_elig_new['Abstract']), custom_ids=list(df_elig_new['custom_id']), instruction=prompt_abstract_individual, model='gpt-4.1-2025-04-14', input_json=False):
        jsonl_file.write(json.dumps(entry, ensure_ascii=False) + "\n")

In [ ]:
create_prediction_batch_json(
        text_summaries = full_texts_elig,
        custom_ids     = list(df_elig_new["custom_id"]),
        instruction    = prompt_full_individual,
        model          = "gpt-4.1-2025-04-14",
        input_json     = False,
    )

1887992

# Predict based on Collection Report

In [60]:
with open(f"openAI_batch_output/synthesis_251111.jsonl", "r", encoding="utf-8") as jsonl_file:
    jsonl_synthesis = [json.loads(line) for line in jsonl_file]  # Load each line as a dictionary

In [61]:
def create_prediction_batch_json_synReport(jsonl_summaries=jsonl_synthesis, model='gpt-4.1-2025-04-14', prompts = prompts_baseline):
    requests = list()
    # first get baseline prompt
    for j in jsonl_summaries:
        augmented_prompts = append_from_text(text=j['response']['body']['choices'][0]['message']['content'], messages=prompts, instruction=prompt_summary)
        for n, p in enumerate(augmented_prompts):
            requests.append(
                {"custom_id": f"{j['custom_id']}/Q{n+1}",
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {"model": model, "messages": [ {"role":"system", "content": system_prompt},
                    {"role": "user",
                "content": p}],
                "logprobs": True,
                "top_logprobs": 20,
                "temperature": 0}}
            )
    return requests

In [62]:
with open(f"OpenAI_batch_input/prediction_251110_report_41.jsonl", "w", encoding="utf-8") as jsonl_file:
    for entry in create_prediction_batch_json_synReport():
        jsonl_file.write(json.dumps(entry, ensure_ascii=False) + "\n")